In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


### 1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.


In [2]:
data = pd.read_csv('data-logistic.csv', header=None)
y = data.iloc[:, 0].to_numpy(dtype=float)
X = data.iloc[:, 1:].to_numpy(dtype=float)


### 2. Убедитесь, что выше выписаны правильные формулы для градиентного спуска. Обратите внимание, что мы используем полноценный градиентный спуск, а не его стохастический вариант!


In [3]:
step = 0.1
eps = 1e-5
max_iter = 10000
initial_w = np.array([0.0, 0.0])


### 3. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляризации 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).


In [4]:
def gradient_descent(C):
    w = initial_w.copy()
    for _ in range(max_iter):
        w_old = w.copy()
        margin = y * (X @ w)
        gradient = ((y[:, None] * X) * (1 - 1 / (1 + np.exp(-margin)))[:, None]).mean(axis=0) - C * w
        w = w + step * gradient
        if np.linalg.norm(w - w_old) <= eps:
            break
    return w

w_without_regularization = gradient_descent(C=0.0)
w_with_regularization = gradient_descent(C=10.0)


### 4. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5). Рекомендуется ограничить сверху число итераций десятью тысячами.


In [5]:
def sigmoid(values):
    return 1 / (1 + np.exp(-values))

scores_without_regularization = sigmoid(X @ w_without_regularization)
scores_with_regularization = sigmoid(X @ w_with_regularization)


### 5. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании? Эти величины будут ответом на задание. В качестве ответа приведите два числа через пробел. Обратите внимание, что на вход функции roc_auc_score нужно подавать оценки вероятностей, подсчитанные обученным алгоритмом. Для этого воспользуйтесь сигмоидной функцией: a(x) = 1 / (1 + exp(-w1x1 - w2x2)).


In [6]:
auc_without = roc_auc_score(y, scores_without_regularization)
auc_with = roc_auc_score(y, scores_with_regularization)
print(round(auc_without, 3), round(auc_with, 3))


0.927 0.936


### 6. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги? Как меняется число итераций при уменьшении длины шага?


In [7]:
step_experiments = []
for candidate_step in [2.0, 1.0, 0.5, 0.1, 0.05, 0.01]:
    w = np.array([0.0, 0.0])
    converged = False
    for iteration in range(1, max_iter + 1):
        w_old = w.copy()
        margin = y * (X @ w)
        gradient = ((y[:, None] * X) * (1 - 1 / (1 + np.exp(-margin)))[:, None]).mean(axis=0)
        w = w + candidate_step * gradient
        if np.linalg.norm(w - w_old) <= eps:
            converged = True
            break
    auc = roc_auc_score(y, sigmoid(X @ w))
    step_experiments.append((candidate_step, converged, iteration, auc))

print('step converged iterations AUC')
for step_value, converged, iterations, auc in step_experiments:
    print(f'{step_value:g} {converged} {iterations} {auc:.3f}')
print('Ответ: при слишком большом шаге алгоритм может не сходиться; при уменьшении шага число итераций растет.')


step converged iterations AUC
2 False 10000 0.828
1 True 32 0.927
0.5 True 60 0.927
0.1 True 244 0.927
0.05 True 431 0.927
0.01 True 1479 0.927
Ответ: при слишком большом шаге алгоритм может не сходиться; при уменьшении шага число итераций растет.


### 7. Попробуйте менять начальное приближение. Влияет ли оно на что-нибудь?


In [8]:
initialization_experiments = []
for start in [np.array([0.0, 0.0]), np.array([1.0, 1.0]), np.array([-1.0, 2.0]), np.array([10.0, -10.0])]:
    w = start.astype(float).copy()
    converged = False
    for iteration in range(1, max_iter + 1):
        w_old = w.copy()
        margin = y * (X @ w)
        gradient = ((y[:, None] * X) * (1 - 1 / (1 + np.exp(-margin)))[:, None]).mean(axis=0)
        w = w + 0.1 * gradient
        if np.linalg.norm(w - w_old) <= eps:
            converged = True
            break
    auc = roc_auc_score(y, sigmoid(X @ w))
    initialization_experiments.append((start, converged, iteration, w, auc))

print('start converged iterations w1 w2 AUC')
for start, converged, iterations, weights, auc in initialization_experiments:
    print(f'{start.tolist()} {converged} {iterations} {weights[0]:.3f} {weights[1]:.3f} {auc:.3f}')
print('Ответ: начальное приближение почти не влияет на итоговое качество и веса, но влияет на число итераций до сходимости.')


start converged iterations w1 w2 AUC
[0.0, 0.0] True 244 0.288 0.092 0.927
[1.0, 1.0] True 230 0.288 0.092 0.927
[-1.0, 2.0] True 368 0.288 0.092 0.927
[10.0, -10.0] True 739 0.288 0.091 0.927
Ответ: начальное приближение почти не влияет на итоговое качество и веса, но влияет на число итераций до сходимости.
